# Task 8: RAG with Llama 3.2

A local Retrieval-Augmented Generation system for an academic document.

In [ ]:
%pip install -q pypdf sentence-transformers scikit-learn ollama

In [ ]:
from pathlib import Path
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ollama

MODEL = 'llama3.2'
DOCUMENT_PATH = Path('academic_document.pdf')

def load_document(path):
    if path.exists():
        return '\n'.join((p.extract_text() or '') for p in PdfReader(str(path)).pages)
    return 'Academic sample: students need at least 75 percent attendance for semester examinations. The curriculum includes programming, data structures, databases, operating systems, mathematics, and artificial intelligence.'

text = load_document(DOCUMENT_PATH)
print('Characters:', len(text))

In [ ]:
def chunk_text(text, size=800, overlap=150):
    text = ' '.join(text.split())
    return [text[i:i+size] for i in range(0, len(text), size-overlap)]

chunks = chunk_text(text)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vectors = embedder.encode(chunks, normalize_embeddings=True)
print('Chunks:', len(chunks))

In [ ]:
question = input('Enter your academic question: ').strip()
if not question:
    question = 'What is the minimum attendance required?'

q_vector = embedder.encode([question], normalize_embeddings=True)
scores = cosine_similarity(q_vector, vectors)[0]
indices = np.argsort(scores)[::-1][:3]
retrieved = [(int(i), float(scores[i]), chunks[i]) for i in indices]
context = '\n\n'.join(f'[Score {score:.3f}] {chunk}' for _, score, chunk in retrieved)

prompt = f'''Answer only from the context. If the answer is missing, say so.
Context:
{context}

Question: {question}
'''
response = ollama.chat(model=MODEL, messages=[{'role':'user','content':prompt}])
answer = response['message']['content']

print('\nUSER QUESTION\n', question)
print('\nRETRIEVED CONTEXT\n', context)
print('\nFINAL RESPONSE FROM LLAMA 3.2\n', answer)

## Requirements

1. Install and run Ollama.
2. Run `ollama pull llama3.2`.
3. Put an academic PDF named `academic_document.pdf` beside this notebook.
4. Execute cells in order.